In [1]:
using Revise
using MSUC
using Random
using Plots
using Gurobi
using Statistics
const GRB_ENV = Gurobi.Env()
optimizer=() -> Gurobi.Optimizer(GRB_ENV)

Set parameter Username
Set parameter LicenseID to value 2658456
Academic license - for non-commercial use only - expires 2026-04-28


#11 (generic function with 1 method)

In [2]:
T = 24
graph=MSUC.LinearGraph(T+1);

In [3]:
instance=MSUC.parse_nc4("Data/T-Ramp/10_0_1_w.nc4",  optimizer, 24);1

1

In [49]:
resSP=MSUC.benders_callback(instance, MSUC.extended_BD_FH_OH, silent=false, force=0.0, S=1, batch=1, gap=0.1, timelimit=120);

Set parameter Threads to value 1
Set parameter TimeLimit to value 1.1999998211860657e+02
Set parameter LazyConstraints to value 1
Set parameter MIPGap to value 0.001
Set parameter LazyConstraints to value 1
Set parameter MIPGap to value 0.001
Set parameter Threads to value 1
Set parameter TimeLimit to value 1.1999998211860657e+02
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Core(TM) Ultra 9 185H, instruction set [SSE2|AVX|AVX2]
Thread count: 22 physical cores, 22 logical processors, using up to 1 threads

Non-default parameters:
TimeLimit  1.1999998211860657e+02
MIPGap  0.001
Threads  1
LazyConstraints  1

Optimize a model with 1947 rows, 2701 columns and 32874 nonzeros
Model fingerprint: 0x0c685470
Variable types: 1971 continuous, 730 integer (730 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+03]
  Objective range  [1e+00, 1e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]
(105246.7793440

In [26]:
force = 0.0
model = MSUC.PolicyGraph(
    MSUC.subproblem_builder_UC_extended,
    instance,
    force,
    graph,
    MSUC.ExtendedFormulation,
    sense = :Min,
    lower_bound = 0.0,
    optimizer = optimizer,
)

A policy graph with 25 nodes.
 Node indices: 1, ..., 25


In [133]:
model[2].demand_constraint

1-element Vector{ConstraintRef}:
 power[1]_out + power[2]_out + power[3]_out + power[4]_out + power[5]_out + power[6]_out + power[7]_out + power[8]_out + power[9]_out + power[10]_out + power_shedding[1] - power_curtailement[1] - 953.170832 error_forecast[1] = 953.170832

In [9]:
MSUC.train(model; iteration_limit = 3)

-------------------------------------------------------------------
         MSUC.jl (c) Oscar Dowson and contributors, 2017-25
-------------------------------------------------------------------
problem
  nodes           : 25
  state variables : 740
  scenarios       : 1.00000e+00
  existing cuts   : false
options
  solver          : serial mode
  risk measure    : MSUC.Expectation()
  sampling scheme : MSUC.InSampleMonteCarlo
subproblem structure
  VariableRef                             : [5417, 5421]
  AffExpr in MOI.EqualTo{Float64}         : [1451, 2963]
  AffExpr in MOI.GreaterThan{Float64}     : [1, 1979]
  AffExpr in MOI.LessThan{Float64}        : [776, 3976]
  VariableRef in MOI.EqualTo{Float64}     : [1, 1]
  VariableRef in MOI.GreaterThan{Float64} : [4676, 4678]
  VariableRef in MOI.LessThan{Float64}    : [740, 740]
  VariableRef in MOI.ZeroOne              : [730, 730]
numerical stability report
  matrix range     [1e+00, 1e+03]
  objective range  [1e+00, 3e+03]
  bounds r

In [48]:
MSUC.train(model; iteration_limit = 10)

-------------------------------------------------------------------
         MSUC.jl (c) Oscar Dowson and contributors, 2017-25
-------------------------------------------------------------------
problem


┌ Warning: Re-training a model with existing cuts!
│ 
│ Are you sure you want to do this? The output from this training may be
│ misleading because the policy is already partially trained.
│ 
│ If you meant to train a new policy with different settings, you must
│ build a new model.
│ 
│ If you meant to refine a previously trained policy, turn off this
│ warning by passing `add_to_existing_cuts = true` as a keyword argument
│ to `MSUC.train`.
│ 
│ In a future release, this warning may turn into an error.
└ @ MSUC /home/mathis/Documents/MSUC/src/algorithm.jl:1351


  nodes           : 25
  state variables : 740
  scenarios       : 1.00000e+00
  existing cuts   : true
options
  solver          : serial mode
  risk measure    : MSUC.Expectation()
  sampling scheme : MSUC.InSampleMonteCarlo
subproblem structure
  VariableRef                             : [5417, 5421]
  AffExpr in MOI.EqualTo{Float64}         : [1451, 2963]
  AffExpr in MOI.GreaterThan{Float64}     : [21, 1999]
  AffExpr in MOI.LessThan{Float64}        : [776, 3976]
  VariableRef in MOI.EqualTo{Float64}     : [740, 741]
  VariableRef in MOI.GreaterThan{Float64} : [4676, 4678]
  VariableRef in MOI.LessThan{Float64}    : [740, 740]
  VariableRef in MOI.ZeroOne              : [730, 730]
numerical stability report
  matrix range     [9e-16, 1e+07]
  objective range  [1e+00, 3e+03]
  bounds range     [1e+00, 3e+02]
  rhs range        [1e+00, 2e+07]
  - matrix range contains small coefficients
  - matrix range contains large coefficients
  - rhs range contains large coefficients
Very large

In [45]:
modelmaster = model[1].subproblem
optimize!(modelmaster)
objective_value(modelmaster)
res_gamma = Dict{Tuple{Int, Int,Int}, Float64}()
for unit in instance.Thermalunits
    for (a,b) in unit.intervals
        res_gamma[unit.name, a, b] = value(model[1].states_gamma[unit.name, a, b])
        if res_gamma[unit.name, a, b] > 0
            println((unit.name, a, b, res_gamma[unit.name, a, b]))
        end
    end
end

(1, 6, 25, 1.0)
(2, -3, 25, 1.0)
(3, 1, 25, 1.0)
(4, -1, 25, 1.0)
(5, 1, 25, 1.0)
(6, 0, 14, 1.0)
(7, 0, 25, 1.0)
(8, -5, 3, 1.0)
(9, 9, 24, 1.0)
(10, -8, 25, 1.0)


In [ ]:
MSUC.bin_extensive_neutral(instance; silent=false,  force=0.0, S=1, batch=1, gap=0.001, timelimit=10, res_gamma=res_gamma)

Set parameter TimeLimit to value 10
Set parameter Threads to value 1
Set parameter MIPGap to value 0.001
Set parameter MIPGap to value 0.001
Set parameter Threads to value 1
Set parameter TimeLimit to value 10
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Core(TM) Ultra 9 185H, instruction set [SSE2|AVX|AVX2]
Thread count: 22 physical cores, 22 logical processors, using up to 1 threads

Non-default parameters:
TimeLimit  10
MIPGap  0.001
Threads  1

Optimize a model with 2242 rows, 1057 columns and 9561 nonzeros
Model fingerprint: 0x8b769172
Variable types: 327 continuous, 730 integer (730 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+03]
  Objective range  [1e+00, 1e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+03]
Presolve removed 849 rows and 237 columns
Presolve time: 0.04s
Presolved: 1393 rows, 820 columns, 6850 nonzeros
Variable types: 273 continuous, 547 integer (547 binary)
Found heu

("Data/T-Ramp/10_0_1_w.nc4", 0.1484692096710205, 1.8500531746090138e6, 1.8498382466747123e6, JuMP.Containers.DenseAxisArray{Float64, 2, Ax, L} where {Ax, L<:Tuple{JuMP.Containers._AxisLookup, JuMP.Containers._AxisLookup}}[2-dimensional DenseAxisArray{Float64,2,...} with index sets:
    Dimension 1, [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
    Dimension 2, 0:24
And data, a 10×25 Matrix{Float64}:
 0.0  0.0  0.0  0.0  1.0  1.0  1.0  1.0  …  1.0  1.0  1.0  1.0  1.0  1.0  1.0
 1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0     1.0  1.0  1.0  1.0  1.0  1.0  1.0
 0.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0     1.0  1.0  1.0  1.0  1.0  1.0  1.0
 1.0  1.0  1.0  1.0  0.0  0.0  0.0  0.0     1.0  1.0  1.0  1.0  1.0  1.0  1.0
 0.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0     1.0  1.0  1.0  1.0  1.0  1.0  1.0
 1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  …  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0     1.0  1.0  1.0  1.0  1.0  1.0  0.0
 1.0  1.0  1.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0

In [66]:
model[23].improved_cuts[2].intercept

1.5235667384000001e6

In [121]:
model[1].improved_cuts[end-1].coefficients_gamma[2,-3,3]

0.0

In [36]:
model[1].bellman_function.global_theta.cuts[1].constraint_ref

61211.39751433136 power[1]_out + 15828.956592000002 power[3]_out + 15706.967567999995 power[5]_out + 1273.832804 power[6]_out + 640.782625 power[7]_out + 94279.61420737603 power[9]_out + 1291.306312 power[10]_out - 63691.640199999994 is_on[6,0]_out - 39728.522750000004 is_on[7,0]_out - 108469.730208 is_on[10,0]_out + 4.529876509062255e6 is_on[1,1]_out + 1.0863777133380782e6 is_on[3,1]_out + 1.0869890988013744e6 is_on[5,1]_out + 1.6462939075136011e7 is_on[9,1]_out + 4.48126858504147e6 is_on[1,2]_out + 1.0411119752823251e6 is_on[3,2]_out + 1.0416978863513172e6 is_on[5,2]_out - 768443.3006799996 is_on[8,2]_out + 1.6351390904678935e7 is_on[9,2]_out + 4.432660661020685e6 is_on[1,3]_out + 984433.9702902733 is_on[2,3]_out + 3.7117905205717674e6 is_on[3,3]_out + 9.022318770791132e6 is_on[4,3]_out + 2.853346384353608e6 is_on[5,3]_out + 1.6645819403222867e6 is_on[8,3]_out + 1.6239842734221859e7 is_on[9,3]_out + 4.3840527369999e6 is_on[1,4]_out + 939686.9716407154 is_on[2,4]_out + 8.9729286053469

In [25]:
using JuMP
using Gurobi

# --- Création du modèle ---
model = Model(Gurobi.Optimizer)

# --- Variables ---
@variable(model, x >= 0)          # quantité du produit A
@variable(model, y >= 0)          # quantité du produit B
@variable(model, z<=1)          # 1 si la machine est allumée

JuMP.fix(z, 1.0; force = true)  # forcer la machine à être allumée

# --- Données ---
profit_A = 3.0
profit_B = 5.0
hours_A  = 1.0
hours_B  = 2.0
capacity = 6.0
bigM     = 6.0   # permet de lier la contrainte à la variable binaire

# --- Contraintes ---
@constraint(model, hours_A * x + hours_B * y <= capacity * z)
@constraint(model, x <= 4)    # limite sur la production de A
@constraint(model, y <= 3)    # limite sur la production de B

# --- Objectif ---
@objective(model, Max, profit_A * x + profit_B * y*z - 2 * z)  # -2 = coût d’activation

# --- Résolution ---
optimize!(model)

# --- Résultats ---
println("x = ", value(x))
println("y = ", value(y))
println("z = ", value(z))
println("Profit = ", objective_value(model))


Set parameter Username
Set parameter LicenseID to value 2658456
Academic license - for non-commercial use only - expires 2026-04-28
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Core(TM) Ultra 9 185H, instruction set [SSE2|AVX|AVX2]
Thread count: 22 physical cores, 22 logical processors, using up to 22 threads

Optimize a model with 3 rows, 3 columns and 5 nonzeros
Model fingerprint: 0x10b66120
Model has 1 quadratic objective term
Coefficient statistics:
  Matrix range     [1e+00, 6e+00]
  Objective range  [2e+00, 3e+00]
  QObjective range [1e+01, 1e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [3e+00, 4e+00]
Presolve removed 2 rows and 1 columns
Presolve time: 0.00s
Presolved: 1 rows, 2 columns, 2 nonzeros
Ordering time: 0.00s

Barrier statistics:
 AA' NZ     : 0.000e+00
 Factor NZ  : 1.000e+00
 Factor Ops : 1.000e+00 (less than 1 second per iteration)
 Threads    : 1

                  Objective                Resid

In [23]:
println(model)

Max 5 z*y + 3 x - 2 z
Subject to
 x + 2 y - 6 z ≤ 0
 x ≤ 4
 y ≤ 3
 z = 1
 x ≥ 0
 y ≥ 0



In [24]:
optimize!(model)

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Core(TM) Ultra 9 185H, instruction set [SSE2|AVX|AVX2]
Thread count: 22 physical cores, 22 logical processors, using up to 22 threads

Optimize a model with 3 rows, 3 columns and 5 nonzeros
Model fingerprint: 0x10b66120
Model has 1 quadratic objective term
Coefficient statistics:
  Matrix range     [1e+00, 6e+00]
  Objective range  [2e+00, 3e+00]
  QObjective range [1e+01, 1e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [3e+00, 4e+00]
Presolve removed 2 rows and 1 columns
Presolve time: 0.00s
Presolved: 1 rows, 2 columns, 2 nonzeros
Ordering time: 0.00s

Barrier statistics:
 AA' NZ     : 0.000e+00
 Factor NZ  : 1.000e+00
 Factor Ops : 1.000e+00 (less than 1 second per iteration)
 Threads    : 1

                  Objective                Residual
Iter       Primal          Dual         Primal    Dual     Compl     Time
   0   2.24280436e+01  2.02595226e+01  3.28e+00 1.80e+